# Clasificación multiclase de anomalías TLM:UAV

## Etapa 1: reconstrucción de vuelos y alineación multisensor

Este cuaderno comienza por establecer la unidad estadística y la correspondencia temporal de los sensores. Todavía no entrena modelos: una arquitectura sofisticada no puede corregir una tabla cuya procedencia o alineación sean ambiguas.

## 1. Alcance y criterio metodológico

La tabla publicada `Fusion_Data.csv` no se utiliza como matriz de entrenamiento. Los vuelos se reconstruyen desde GPS, VIBE, RATE e IMU sin consultar sus etiquetas. Después se adopta el reloj de VIBE como rejilla canónica de aproximadamente 10 Hz y se alinean únicamente observaciones disponibles en el mismo vuelo y no posteriores al instante de referencia.

Los labels de otros sensores se conservan en una tabla de auditoría separada. Esta separación impide que entren accidentalmente como predictores y permite medir discrepancias en los bordes de los episodios.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.audit import dataset_manifest
from src.data import (
    FlightSplitConfig,
    align_multisensor_data,
    available_feature_views,
    make_reconstruction_config,
    persist_data_protocol,
    persist_reconstruction_artifacts,
    prepare_lofo_data,
    reconstruct_flights,
    verify_external_test_isolation,
    verify_label_independent_flight_ids,
)

pd.set_option("display.max_columns", 120)

## 2. Configuración central

Las tolerancias son explícitas. GPS se admite hasta 220 ms hacia el pasado, ligeramente por encima de su periodo nominal de 200 ms; IMU se admite hasta 45 ms, ligeramente por encima de sus 40 ms. No se permite búsqueda hacia el futuro ni relleno sin límite a través de gaps.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the TLM project root containing dataset/."
    )

RESULTS_DIR = PROJECT_ROOT / "results" / "reconstruction"
PROTOCOL_RESULTS_DIR = PROJECT_ROOT / "results" / "protocol"
RECONSTRUCTION_CONFIG = make_reconstruction_config(PROJECT_ROOT)
SPLIT_CONFIG = FlightSplitConfig(
    validation_fraction=0.20,
    purge_seconds=5.0,
    transition_guard_seconds=0.25,
)

{
    "dataset_dir": RECONSTRUCTION_CONFIG.dataset_dir.name,
    "gps_gap_threshold_ms": RECONSTRUCTION_CONFIG.gps_gap_threshold_ms,
    "gps_alignment_tolerance_us": (
        RECONSTRUCTION_CONFIG.gps_alignment_tolerance_us
    ),
    "imu_alignment_tolerance_us": (
        RECONSTRUCTION_CONFIG.imu_alignment_tolerance_us
    ),
    "alignment_direction": RECONSTRUCTION_CONFIG.alignment_direction,
    "target_source": RECONSTRUCTION_CONFIG.target_source,
}

{'dataset_dir': 'dataset',
 'gps_gap_threshold_ms': 600000,
 'gps_alignment_tolerance_us': 220000,
 'imu_alignment_tolerance_us': 45000,
 'alignment_direction': 'backward',
 'target_source': 'VIBE'}

## 3. Reconstrucción de `flight_id`

GPS contiene semana y milisegundos GPS absolutos, por lo que sus dos sesiones se separan directamente. VIBE conserva exactamente el primer archivo de referencia y permite identificar el vuelo 0 mediante valores no relacionados con el target. RATE comparte timestamps con VIBE y se asigna por proximidad del número de línea. IMU se compara con ambas trayectorias y utiliza las coincidencias únicas de sus seis señales en la tabla fusionada solo como evidencia de procedencia.

In [3]:
manifest_before = dataset_manifest(RECONSTRUCTION_CONFIG.dataset_dir)
reconstructed = reconstruct_flights(RECONSTRUCTION_CONFIG)

display(reconstructed.assignment_summary)
display(pd.Series(reconstructed.imu_anchor_summary, name="IMU anchor audit"))
print("Reconstruction fingerprint:", reconstructed.fingerprint)

,source,flight_id,method,rows,time_min,time_max,line_min,line_max,time_is_strictly_increasing,line_is_strictly_increasing,labels_used_for_assignment,minimum_line_delta,maximum_line_delta,assignment_ties,minimum_assignment_margin,rows_with_margin_below_100,fusion_anchor_rows,fusion_anchor_overrides
0,GPS,0,absolute_gps_gap,2450,15403836,521403855,1760,439159,True,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GPS,1,absolute_gps_gap,1869,22403535,434803509,1752,335467,True,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,VIBE,0,exact_nonlabel_lineage,4900,15363852,521463831,1736,439221,True,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,VIBE,1,exact_nonlabel_lineage,3738,22363551,434863485,1729,335531,True,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RATE,0,exact_time_nearest_vibe_line,4900,15363852,521463831,1715,439196,True,True,False,20.0,30.0,0.0,NaN,NaN,NaN,NaN
5,RATE,1,exact_time_nearest_vibe_line,3738,22363551,434863485,1708,335504,True,True,False,20.0,30.0,0.0,NaN,NaN,NaN,NaN
6,IMU,0,vibe_trajectory_with_unique_fusion_anchor,12253,15303876,521503815,1676,439242,True,True,False,NaN,NaN,NaN,4.679424,4.0,12065.0,1.0
7,IMU,1,vibe_trajectory_with_unique_fusion_anchor,9344,22343559,434863485,1687,335534,True,True,False,NaN,NaN,NaN,4.679424,4.0,12065.0,1.0


unique_anchor_rows            12065
dominant_flight                   0
dominant_flight_purity     0.999917
overridden_rows                   1
anchor_time_min            15303876
anchor_time_max           521503815
anchor_time_monotonic          True
labels_used                   False
Name: IMU anchor audit, dtype: object

Reconstruction fingerprint: 90753590ef5e761634b7aeb7dbf8172d9a8bff7352a93c0c2e9114001f9628ba


## 4. Evidencia de asignación y ausencia de target leakage

Una asignación de vuelo no debe cambiar si se alteran los labels. La siguiente prueba permuta en memoria todas las columnas de etiqueta, reconstruye de nuevo los cuatro sensores y exige exactamente los mismos `flight_id`. También comprueba los conteos y la monotonicidad temporal esperados.

In [4]:
expected_counts = {
    "GPS": {0: 2450, 1: 1869},
    "RATE": {0: 4900, 1: 3738},
    "VIBE": {0: 4900, 1: 3738},
    "IMU": {0: 12253, 1: 9344},
}
observed_counts = {
    source: frame["flight_id"].value_counts().sort_index().to_dict()
    for source, frame in reconstructed.frames.items()
}

assert observed_counts == expected_counts
assert reconstructed.assignment_summary[
    "time_is_strictly_increasing"
].all()
assert reconstructed.assignment_summary[
    "line_is_strictly_increasing"
].all()
assert not reconstructed.assignment_summary[
    "labels_used_for_assignment"
].any()
assert verify_label_independent_flight_ids(reconstructed, seed=42)

observed_counts

{'GPS': {0: 2450, 1: 1869},
 'RATE': {0: 4900, 1: 3738},
 'VIBE': {0: 4900, 1: 3738},
 'IMU': {0: 12253, 1: 9344}}

## 5. Alineación causal sobre el reloj VIBE

RATE coincide exactamente con los instantes VIBE. Para GPS e IMU se selecciona la última observación disponible dentro del mismo vuelo. Cuatro filas no tienen un GPS previo dentro de la tolerancia: la primera fila y la primera fila posterior al gran gap de cada vuelo. Se descartan explícitamente en lugar de usar un dato futuro o propagar un estado obsoleto.

In [5]:
aligned = align_multisensor_data(reconstructed)

print("Aligned shape:", aligned.frame.shape)
print("Alignment fingerprint:", aligned.alignment_fingerprint)
display(aligned.alignment_summary)
display(aligned.dropped_rows)

Aligned shape: (8634, 48)
Alignment fingerprint: 171b7e4375d97cb852e43ad2fa81faceac9f01017146acdd8cf1324d0fa869c6


,flight_id,source,aligned_rows,unique_source_rows,maximum_reuse,lag_min_us,lag_median_us,lag_p99_us,lag_max_us
0,0,GPS,4898,2450,2,59976,60809.0,160769.0,160769
1,0,RATE,4898,4898,1,0,0.0,0.0,0
2,0,VIBE,4898,4898,1,0,0.0,0.0,0
3,0,IMU,4898,4898,1,0,0.0,19992.0,20825
4,1,GPS,3736,1869,2,59976,60809.0,160769.0,160769
5,1,RATE,3736,3736,1,0,0.0,0.0,0
6,1,VIBE,3736,3736,1,0,0.0,0.0,0
7,1,IMU,3736,3736,1,0,0.0,19992.0,20825


,flight_id,time_us,episode_id,label,missing_sources
0,0,15363852,0,0,GPS
1,0,45963274,0,0,GPS
2,1,22363551,0,0,GPS
3,1,75563929,0,0,GPS


## 6. Referencia explícita para las etiquetas

El target es `VIBE.labels` porque VIBE define la fila canónica. Esta es una convención reproducible, no una afirmación de que VIBE sea una fuente de verdad física superior. Los labels alineados de GPS, RATE e IMU se comparan únicamente para cuantificar desacuerdos, que se concentran alrededor de transiciones.

In [6]:
class_distribution = (
    aligned.frame.groupby(["flight_id", "label"])
    .size()
    .rename("rows")
    .reset_index()
)
display(class_distribution)
display(aligned.label_agreement)

,flight_id,label,rows
0,0,0,2460
1,0,1,609
2,0,2,594
3,0,3,482
4,0,4,753
5,1,0,1595
6,1,1,529
7,1,2,643
8,1,3,346
9,1,4,623


,flight_id,target_source,compared_source,rows,disagreements,disagreement_rate
0,0,VIBE,GPS,4898,11,0.002246
1,0,VIBE,RATE,4898,6,0.001225
2,0,VIBE,IMU,4898,3,0.000612
3,1,VIBE,GPS,3736,12,0.003212
4,1,VIBE,RATE,3736,7,0.001874
5,1,VIBE,IMU,3736,1,0.000268


## 7. Verificaciones de integridad y leakage

La tabla destinada al modelado contiene un solo target y ninguna etiqueta auxiliar. Los tiempos, identificadores de vuelo, episodios y `row_id` se conservan como metadatos, pero no pertenecen a `feature_columns`. Todas las uniones deben ser causales, permanecer dentro del vuelo y respetar sus tolerancias.

In [7]:
assert aligned.checks.all(), aligned.checks[~aligned.checks]
assert set(aligned.metadata_columns()).isdisjoint(aligned.feature_columns)
assert {"gps_label", "rate_label", "imu_label"}.isdisjoint(
    aligned.frame.columns
)
assert manifest_before == dataset_manifest(RECONSTRUCTION_CONFIG.dataset_dir)

display(aligned.checks.rename("passed").to_frame())

,passed
expected_aligned_shape,True
expected_rows_by_flight,True
four_uncovered_rows_reported,True
row_ids_are_unique,True
flight_time_key_is_unique,True
target_values_are_valid,True
all_classes_exist_in_each_flight,True
features_are_finite,True
features_have_no_missing_values,True
source_labels_are_isolated,True


## 8. Inventario de variables alineadas

La alineación conserva toda la telemetría candidata y antepone el sensor al nombre de columna. Esto evita colisiones semánticas. La tabla aún no debe entregarse completa a un modelo: las vistas `sensor_core` y `full_diagnostic`, así como los transformadores ajustados solo con desarrollo, se definirán en la siguiente etapa.

In [8]:
feature_inventory = pd.DataFrame(
    {
        "feature": aligned.feature_columns,
        "source": [
            column.split("__", maxsplit=1)[0].upper()
            for column in aligned.feature_columns
        ],
        "n_unique": [
            aligned.frame[column].nunique()
            for column in aligned.feature_columns
        ],
    }
)
display(feature_inventory)

,feature,source,n_unique
0,gps__I,GPS,1
1,gps__Status,GPS,2
2,gps__NSats,GPS,2
3,gps__HDop,GPS,1
4,gps__Lat,GPS,2646
5,gps__Lng,GPS,2714
6,gps__Alt,GPS,1102
7,gps__Spd,GPS,2543
8,gps__GCrs,GPS,4143
9,gps__VZ,GPS,1347


## 9. Persistencia transparente

Se guardan CSV legibles para la tabla alineada, la trazabilidad temporal, los labels auxiliares y los resúmenes de asignación. El JSON asociado registra configuración, hashes, fingerprints, filas descartadas y verificaciones. Los archivos fuente permanecen inmutables.

In [9]:
artifact_paths = persist_reconstruction_artifacts(
    reconstructed,
    aligned,
    RESULTS_DIR,
)
relative_artifacts = {
    name: path.relative_to(PROJECT_ROOT).as_posix()
    for name, path in artifact_paths.items()
}
relative_artifacts

{'aligned_data': 'results/reconstruction/aligned_multisensor.csv',
 'label_audit': 'results/reconstruction/aligned_label_audit.csv',
 'alignment_audit': 'results/reconstruction/alignment_trace.csv',
 'assignment_summary': 'results/reconstruction/flight_assignment_summary.csv',
 'alignment_summary': 'results/reconstruction/alignment_summary.csv',
 'metadata': 'results/reconstruction/reconstruction_metadata.json'}

## 10. Cierre de la reconstrucción

La reconstrucción produce dos trayectorias coherentes por sensor y una tabla causal de 8.634 filas. Las cinco clases aparecen en ambos vuelos, por lo que leave-one-flight-out multiclase es técnicamente posible. Sin embargo, cada tipo de falla sigue representado por un solo episodio por vuelo y `flight_id` continúa siendo una identidad inferida, no una anotación oficial.

Los siguientes bloques congelan las vistas de variables, el split interno purgado y el preprocesamiento ajustado exclusivamente con el vuelo de desarrollo. Ninguna métrica de modelo se calcula durante esta preparación.

## 11. Vistas de variables congeladas

Las vistas se definen por significado físico y disponibilidad operacional, no mediante correlación con el target ni rankings calculados sobre ambos vuelos. `sensor_core` conserva 15 mediciones instantáneas. `full_diagnostic` contiene 29 variables numéricas y tres estados categóricos, incluidos contexto de navegación, referencias y salidas de control.

La segunda vista no se denomina libre de leakage: sirve para medir cuánto depende el rendimiento de información diagnóstica o contextual adicional.

In [10]:
feature_views = available_feature_views()
feature_view_table = pd.DataFrame(
    [
        {
            "view": view.name,
            "numerical_features": len(view.numerical_columns),
            "categorical_features": len(view.categorical_columns),
            "total_features": len(view.columns),
            "description": view.description,
        }
        for view in feature_views
    ]
)
assert all(
    set(view.columns).issubset(aligned.feature_columns)
    for view in feature_views
)
assert all(
    set(view.columns).isdisjoint(aligned.metadata_columns())
    for view in feature_views
)
display(feature_view_table)

,view,numerical_features,categorical_features,total_features,description
0,sensor_core,15,0,15,Instantaneous physical measurements without po...
1,full_diagnostic,29,3,32,Nonconstant operational telemetry including na...


## 12. Folds externos leave-one-flight-out

Cada dirección utiliza un vuelo completo como desarrollo y mantiene el otro fuera de imputación, escalado, codificación, pesos de clase y selección de época. Después se intercambian los vuelos. Estos dos folds son evaluaciones dirigidas sobre dos unidades inferidas, no réplicas poblacionales independientes.

In [11]:
folds, prepared_folds = prepare_lofo_data(
    aligned,
    split_config=SPLIT_CONFIG,
    feature_views=feature_views,
)
fold_summary = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "development_flight": fold.development_flight,
            "test_flight": fold.test_flight,
            **fold.report["sizes"],
            "minimum_gap_seconds": fold.report[
                "minimum_train_valid_gap_seconds"
            ],
            "fingerprint": fold.fingerprint,
        }
        for fold in folds
    ]
)
display(fold_summary)

,fold,development_flight,test_flight,inner_train,validation,purged,development_full,external_test,stable_external_test,transition_external_test,minimum_gap_seconds,fingerprint
0,develop_f0_test_f1,0,1,3132,975,791,4898,3736,3701,35,5.000499,31de5e50eb3ad6f34cdd95ae578e801afc0549da8a67e2...
1,develop_f1_test_f0,1,0,2198,744,794,3736,4898,4863,35,5.000499,a72f7ccf7cd2e7870b5ba7fdfc7607c2e026fb03a13a5f...


## 13. Validación interna purgada

Cada clase de falla aparece en un solo episodio por vuelo. Una validación cronológica única dejaría clases fuera de train o validación. Para seleccionar época, se reserva el bloque central del 20 % de cada uno de los ocho episodios y se retiran cinco segundos a cada lado del entrenamiento interno.

Esta validación comparte episodio con train y solo es una herramienta de optimización. La evidencia de transferencia procede exclusivamente del vuelo externo.

In [12]:
for fold in folds:
    print(f"\n{fold.name}")
    display(pd.DataFrame(fold.report["class_counts"]))
    display(pd.Series(fold.report["checks"], name="passed"))
    assert all(fold.report["checks"].values())
    assert fold.report["test_used_for_preprocessing_or_selection"] is False


develop_f0_test_f1


,inner_train,validation,development_full,external_test,stable_external_test
0,1573,490,2460,1595,1578
1,390,121,609,529,524
2,378,118,594,643,638
3,287,96,482,346,341
4,504,150,753,623,620


development_and_test_flights_differ        True
development_and_test_rows_are_disjoint     True
inner_roles_are_disjoint                   True
inner_roles_partition_development          True
test_is_not_an_inner_role                  True
all_classes_in_inner_train                 True
all_classes_in_validation                  True
all_classes_in_external_test               True
all_classes_in_stable_test                 True
purge_is_at_least_configured_duration      True
stable_and_transition_test_are_disjoint    True
test_masks_partition_external_test         True
expected_episode_count                     True
Name: passed, dtype: bool


develop_f1_test_f0


,inner_train,validation,development_full,external_test,stable_external_test
0,879,318,1595,2460,2443
1,325,105,529,609,604
2,416,128,643,594,589
3,178,69,346,482,477
4,400,124,623,753,750


development_and_test_flights_differ        True
development_and_test_rows_are_disjoint     True
inner_roles_are_disjoint                   True
inner_roles_partition_development          True
test_is_not_an_inner_role                  True
all_classes_in_inner_train                 True
all_classes_in_validation                  True
all_classes_in_external_test               True
all_classes_in_stable_test                 True
purge_is_at_least_configured_duration      True
stable_and_transition_test_are_disjoint    True
test_masks_partition_external_test         True
expected_episode_count                     True
Name: passed, dtype: bool

## 14. Máscara secundaria de transiciones

La auditoría mostró que todos los desacuerdos entre sensores están a 0,2 s o menos de una transición VIBE. La evaluación primaria conservará todas las filas externas. Como sensibilidad secundaria, queda congelada una banda de ±0,25 s alrededor de cada cambio de label. Esta máscara nunca interviene en entrenamiento, early stopping ni ranking principal.

In [13]:
transition_summary = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "external_rows": len(fold.test_row_ids),
            "stable_rows": len(fold.stable_test_row_ids),
            "guard_rows": len(fold.transition_test_row_ids),
            "guard_fraction": (
                len(fold.transition_test_row_ids) / len(fold.test_row_ids)
            ),
        }
        for fold in folds
    ]
)
display(transition_summary)

,fold,external_rows,stable_rows,guard_rows,guard_fraction
0,develop_f0_test_f1,3736,3701,35,0.009368
1,develop_f1_test_f0,4898,4863,35,0.007146


## 15. Preprocesamiento en dos fases

Durante early stopping, medianas, medias, desviaciones, constantes y categorías se ajustan solo con `inner_train`. Tras seleccionar la época, se crea un estado nuevo con todo el vuelo de desarrollo para el refit final. El vuelo externo únicamente recibe `transform`.

Las categorías reservan el índice 0 para valores desconocidos. Los pesos balanceados se calculan por separado con inner-train y con desarrollo completo; no se aplica sobremuestreo.

In [14]:
prepared_summary = pd.DataFrame(
    [
        {
            "fold": item.fold.name,
            "view": item.feature_view.name,
            "train_num_shape": item.train.X_num.shape,
            "train_cat_shape": item.train.X_cat.shape,
            "valid_rows": len(item.valid.row_ids),
            "development_rows": len(item.development.row_ids),
            "test_rows": len(item.test.row_ids),
            "tuning_cardinalities": (
                item.tuning_state.categorical_cardinalities
            ),
            "dropped_tuning_columns": (
                item.tuning_state.dropped_numerical_columns
            ),
            "unknown_test_categories": sum(
                item.test.unknown_category_counts.values()
            ),
        }
        for item in prepared_folds
    ]
)
display(prepared_summary)

,fold,view,train_num_shape,train_cat_shape,valid_rows,development_rows,test_rows,tuning_cardinalities,dropped_tuning_columns,unknown_test_categories
0,develop_f0_test_f1,sensor_core,"(3132, 15)","(3132, 0)",975,4898,3736,(),(),0
1,develop_f0_test_f1,full_diagnostic,"(3132, 29)","(3132, 3)",975,4898,3736,"(3, 3, 3)",(),0
2,develop_f1_test_f0,sensor_core,"(2198, 15)","(2198, 0)",744,3736,4898,(),(),0
3,develop_f1_test_f0,full_diagnostic,"(2198, 29)","(2198, 3)",744,3736,4898,"(3, 3, 3)",(),0


## 16. Verificaciones del protocolo

Las comprobaciones siguientes validan índices, soporte de clases, normalización, cardinalidades y procedencia de cada estado. La preparación completa se repite para exigir fingerprints idénticos.

In [15]:
for item in prepared_folds:
    assert item.checks.all(), item.checks[~item.checks]

assert verify_external_test_isolation(
    aligned,
    folds,
    feature_views,
)

repeated_folds, repeated_prepared = prepare_lofo_data(
    aligned,
    split_config=SPLIT_CONFIG,
    feature_views=feature_views,
)
assert [fold.fingerprint for fold in folds] == [
    fold.fingerprint for fold in repeated_folds
]
assert [item.fingerprint for item in prepared_folds] == [
    item.fingerprint for item in repeated_prepared
]

integrity_table = pd.DataFrame(
    {
        f"{item.fold.name}/{item.feature_view.name}": item.checks
        for item in prepared_folds
    }
)
display(integrity_table)

,develop_f0_test_f1/sensor_core,develop_f0_test_f1/full_diagnostic,develop_f1_test_f0/sensor_core,develop_f1_test_f0/full_diagnostic
tuning_state_fitted_only_on_inner_train,True,True,True,True
final_state_fitted_only_on_development,True,True,True,True
external_test_absent_from_tuning_fit,True,True,True,True
external_test_absent_from_final_fit,True,True,True,True
validation_absent_from_tuning_fit,True,True,True,True
transformed_row_counts_are_correct,True,True,True,True
all_numerical_arrays_are_finite,True,True,True,True
inner_train_is_standardized,True,True,True,True
development_refit_is_standardized,True,True,True,True
categorical_indices_are_valid,True,True,True,True


## 17. Persistencia del protocolo

Los roles de cada fila, las vistas de variables y los parámetros ajustados se guardan en formatos legibles. No se persisten tuplas posicionales ni transformadores opacos. Estos fingerprints deberán acompañar cada ejecución futura.

In [16]:
protocol_paths = persist_data_protocol(
    aligned,
    folds,
    prepared_folds,
    PROTOCOL_RESULTS_DIR,
)
{
    name: path.relative_to(PROJECT_ROOT).as_posix()
    for name, path in protocol_paths.items()
}

{'fold_assignments': 'results/protocol/fold_assignments.csv',
 'feature_views': 'results/protocol/feature_views.json',
 'preprocessing': 'results/protocol/preprocessing_metadata.json'}

## 18. Datos listos para modelado

La entrada de los modelos queda ahora completamente determinada: misma alineación, folds, vistas, labels, pesos y transformaciones. El test externo no interviene en ninguna operación ajustable.

La limitación permanece intacta: la validación interna comparte episodios y solo hay dos vuelos inferidos. El entrenamiento podrá comparar arquitecturas bajo un protocolo consistente, pero no convertir esta muestra en evidencia poblacional amplia.